In [1]:
import numpy as np
import h5py
from tqdm import tqdm

def process_data_and_get_corr_matrix(num_pairs, v1, v2):
    """
    Process data and return the correlation matrices for non-equal pair numbers for each participant.
    """
    corr_matrices = []

    # Loop over each participant (pair_num corresponds to each participant)
    for pair_num in range(num_pairs):
        send_stack = v1[pair_num]
        
        # Initialize an empty list to collect non-equal pair correlations for the current participant
        corr_matrix_for_current_participant = []

        # Loop over each other pair number (to get non-equal pairs)
        for pair_num_other in range(num_pairs):
            if pair_num != pair_num_other:  # Ensure non-equal pair indices
                rece_stack = v2[pair_num_other]  # Use non-equal pair index from v2
                
                # Concatenate the two conditions
                pair_matrix = np.concatenate((send_stack, rece_stack), axis=0)

                # Calculate the correlation matrix
                corr_matrix = np.corrcoef(pair_matrix)

                # Fisher's Z Transformation
                fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))

                # Extract the part of the matrix that correlates the two conditions
                corr_matrix_half = fisher_z_matrix[:send_stack.shape[0], send_stack.shape[0]:]  
                
                # Append the current non-equal pair correlation to the list
                corr_matrix_for_current_participant.append(corr_matrix_half)

        # Stack all non-equal pair correlations for the current participant
        corr_matrices.append(np.stack(corr_matrix_for_current_participant))

    # Convert the list of stacked correlation matrices into a numpy array
    corr_matrices = np.array(corr_matrices)
    return corr_matrices


def calculate_z_corr_matrix2(corr_matrices_12, corr_matrices_34):
    corr_matrices_34 = np.array(corr_matrices_34)
    corr_matrices_12 = np.array(corr_matrices_12)
    corr_matrix_all = np.concatenate((corr_matrices_12, corr_matrices_34), axis=1)
    return corr_matrix_all

def sliding_window_analysis(data_A, data_B, window_size=2, step_size=1):
    """
    data_A: 时间序列数据 (23被试, 12试次, time_points, voxel)
    data_B: 目标数据 (23被试, 12试次, voxel)
    """

    n_timepoints = data_A.shape[2]
    n_windows = (n_timepoints - window_size + 1) // step_size
    
    all_correlations = []
    
    for w in range(n_windows):
        start_idx = w * step_size
        end_idx = start_idx + window_size
        
        window_mean = np.mean(data_A[..., start_idx:end_idx, :], axis=-2)
        
        corr = process_data_and_get_corr_matrix(23, window_mean, data_B)
        all_correlations.append(corr)
    
    return all_correlations

def load_data(file_path):
    """Function to load data from subarray_0 to subarray_399 from a given file path."""
    with h5py.File(file_path, 'r') as file:
        data_list = []
        for i in range(400):
            dataset_name = f'subarray_{i}'
            if dataset_name in file:
                data = file[dataset_name][:]
                data_list.append(data)
            else:
                print(f"Dataset '{dataset_name}' not found in the file.")
        return data_list


def load_split_data(file_path):
    """
    加载分成f2和f5两组的数据
    file_path: h5文件路径
    返回: f2_list, f5_list, 每个list包含400个脑区的数据
    """
    f2_list = []
    f5_list = []
    
    with h5py.File(file_path, 'r') as file:
        # 加载f2组数据
        f2_group = file['f2']
        # 加载f5组数据
        f5_group = file['f5']
        
        # 获取所有脑区的数据
        for i in range(400):
            region_name = f'region_{i}'
            if region_name in f2_group and region_name in f5_group:
                f2_data = f2_group[region_name][:]
                f5_data = f5_group[region_name][:]
                f2_list.append(f2_data)
                f5_list.append(f5_data)
            else:
                print(f"Region {i} not found in the file.")
                
    return f2_list, f5_list


def check_for_nans(data_list, list_name):
    """Function to check for NaN values in a list of datasets."""
    for i, dataset in enumerate(data_list):
        if np.isnan(dataset).any():
            print(f"NaN found in dataset {i} of {list_name}")

def main():
    # 定义文件路径
    file_path1 = 'face10156_speak_va.h5'
    file_path2 = 'face10156_listen_vb_noavg.h5'
    file_path3 = 'face10156_speak_vb.h5'
    file_path4 = 'face10156_listen_va_noavg.h5'
    # 加载数据
    print("Loading data...")
    # 加载普通格式数据
    stacked_vas_list = load_data(file_path1)
    stacked_vbs_list = load_data(file_path3)
    
    # 加载分组数据
    vbr_f2_list, vbr_f5_list = load_split_data(file_path2)
    var_f2_list, var_f5_list = load_split_data(file_path4)

    all_z_correlations_real = []

    # 主分析循环
    for brain_area in tqdm(range(400), desc="Processing brain areas"):
        # 获取当前脑区的数据
        vas_data = stacked_vas_list[brain_area]
        vbs_data = stacked_vbs_list[brain_area]
        
        # f2部分的相关分析
        real_corr_rbsa_f2 = sliding_window_analysis(vbr_f2_list[brain_area], vas_data[:, :12, :])
        real_corr_rasb_f2 = sliding_window_analysis(var_f2_list[brain_area], vbs_data[:, :12, :])
        print(np.array(real_corr_rbsa_f2).shape)
        
        # f5部分的相关分析
        real_corr_rbsa_f5 = sliding_window_analysis(vbr_f5_list[brain_area], vas_data[:, 12:, :])
        real_corr_rasb_f5 = sliding_window_analysis(var_f5_list[brain_area], vbs_data[:, 12:, :])
        print(np.array(real_corr_rbsa_f5).shape)
        
        # 合并f2和f5的结果
        real_corr_rbsa = real_corr_rbsa_f2 + real_corr_rbsa_f5
        real_corr_rasb = real_corr_rasb_f2 + real_corr_rasb_f5

        print(np.array(real_corr_rbsa).shape)

        # 计算Z相关矩阵
        z_corr_matrix_real = calculate_z_corr_matrix2(real_corr_rbsa, real_corr_rasb)
        print(np.array(z_corr_matrix_real).shape)
        
        all_z_correlations_real.append(z_corr_matrix_real)

    # 转换为numpy数组和保存结果
    results = {
        'z_correlations_real': np.array(all_z_correlations_real)
    }

    print("Saving results...")
    with h5py.File('slidewindow2_pseudocorrelation_slface.h5', 'w') as f:
        for key, value in results.items():
            f.create_dataset(key, data=value)

    print("Analysis completed!")
    return results

if __name__ == "__main__":
    results = main()

Loading data...


Processing brain areas:   0%|                                                                   | 0/400 [00:00<?, ?it/s]/tmp/ipykernel_848/4181772952.py:30: RuntimeWarning: divide by zero encountered in divide
  fisher_z_matrix = 0.5 * np.log((1 + corr_matrix) / (1 - corr_matrix))


(9, 23, 22, 12, 12)


Processing brain areas:   0%|▏                                                          | 1/400 [00:01<06:59,  1.05s/it]

(14, 23, 22, 12, 12)
(23, 23, 22, 12, 12)
(23, 46, 22, 12, 12)
(9, 23, 22, 12, 12)


...

Processing brain areas: 100%|████████████████████████████████████████████████████████▊| 399/400 [06:56<00:00,  1.03it/s]

(14, 23, 22, 12, 12)
(23, 23, 22, 12, 12)
(23, 46, 22, 12, 12)
(9, 23, 22, 12, 12)


Processing brain areas: 100%|█████████████████████████████████████████████████████████| 400/400 [06:57<00:00,  1.04s/it]

(14, 23, 22, 12, 12)
(23, 23, 22, 12, 12)
(23, 46, 22, 12, 12)


Saving results...
Analysis completed!
